# Batch BH fitting workflow

This notebook demonstrates how to run batch BH fits using the `bh_molecule.workflows.batch_fit` helpers (which internally use `bh_molecule.workflows.signal_scan` for auto-detection) on the same data folder used in `10_explore_data.ipynb`.

We will:

- Discover FITS files in the experiment folder.
- Run `run_bh_batch` for a single shot with **automatic signal detection**.
- Optionally compare with manual frame/channel selection.
- Run `run_folder_batch` to process many shots with resume capability.
- Inspect and visualize the saved results.


In [ ]:
from pathlib import Path

from bh_molecule import run_bh_batch, run_folder_batch


def get_fits_files(folder, recursive=False):
    """Return a sorted list of .fits files under folder.

    Mirrors the helper used in `10_explore_data.ipynb`.
    """
    folder = Path(folder)
    if recursive:
        return sorted(folder.rglob("*.fits"))
    return sorted(folder.glob("*.fits"))


# Use the same experiment folder as in 10_explore_data.ipynb
datafolder = Path("~/Dropbox/Experiments/2025-LHD-BH/133mORCA").expanduser()
fits_files = get_fits_files(datafolder)
datafolder, len(fits_files)

In [ ]:
# Inspect the first FITS file we will use for fitting
fits_files[0]

In [ ]:
# Batch-fit configuration
cw_nm = 431.91  # CW used for calibration

# Automatic mode: let the workflow scan for frames/channels with signal
frames = None
channels = None
background_frames = (0, 1, 2, 3)

# Example manual override (uncomment to force specific indices)
# frames = [9, 10, 11]
# channels = [30, 31, 32]

time_range = (0.0, 10.0)  # seconds
out_dir = Path.cwd() / "bh_batch_results"
out_dir

In [ ]:
# Run BH batch fitting for all shots in the folder
# Resume behaviour: shots with existing summary.csv under out_dir/<shot_id>/ are skipped.
# When frames/channels are None, the workflow will scan for signal using
# bh_molecule.workflows.signal_scan under the hood.
results = run_folder_batch(
    datafolder,
    frames=frames,
    channels=channels,
    cw=cw_nm,
    scale=1.0,
    time_range=time_range,
    background_frames=background_frames,
    out_dir=out_dir,
)
list(results.keys())[:5]


In [ ]:
# Example: inspect one of the batch summaries returned by run_folder_batch
if results:
    first_shot_id = sorted(results.keys())[0]
    results[first_shot_id].head()
